# MACHINE LEARNING - CA1 | Exploring Diabetes Risk Prediction through Machine Learning: Clinical and Lifestyle Indicators.

Claudia Victoria Vilchis Mondragon

Diabetes is a major public health problem, affecting 589 million adults worldwide (International Diabetes Federation, 2025). This project uses the Diabetes Health Indicators dataset, based on the CDC’s BRFSS 2015 survey (253,680 responses), to compare machine learning classification models for predicting diabetes risk.

Research questions: 
1. What kind of machine-learning model is best at predicting diabetes risk, and why?
2. Which lifestyle factors are most significant in predicting diabetes risk compared with clinical indicators?
3. How do different test-set sizes (5%, 15% and 25%) affect prediction performance, and how stable are the results under cross-validation?

In [1]:
# Read and import library packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('diabetes_binary_health_indicators_BRFSS2015.csv')

# Data Understanding and Characterisation

In [3]:
df.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


- Observation: I ran df.head() to see the first five rows and get an idea of how the data is organised.

In [4]:
df.shape

(253680, 22)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_binary       253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  MentHlth   

Observation:  
- I checked the dataset using df.shape and df.info(). The dataset has 253,680 rows and 22 columns. Each row represents an individual survey respondent, and the columns contain variables related to diabetes and health.
- There are 21 predicting variables and one target variable whis is "Diabetes_binary" (0= no diabetes, 1=Prediabetes or diabetes).
- All 22 columns have 253,680 no missing values and all the variables are stored as float64.
- There are 3 types of variables in this dataset: binary like Smoker, ordinal like Age and numerical like BMI; even though they're all stored as numbers, each one needs to be treated differently for the visualisation and the analysis.
- This first step helped me understand the structure of the data before moving on to the next stages of the analysis.

In [6]:
df.isnull().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

Observation: Confirming that there are no missing values, so I don't need to apply imputation

In [7]:
df.duplicated().sum()

np.int64(24206)

Obervation: I can see that the dataset has 24,206 duplicate rows, and mostly the variables are binary, so some duplicates may correspond to different respondents but with identical answers. However, if I keep them, I could exaggerate the results as the same rows might appear in the training and test sets. That is why I have to removed.

In [11]:
df = df.drop_duplicates()

In [13]:
df.shape

(229474, 22)

In [14]:
df["Diabetes_binary"].value_counts()

Diabetes_binary
0.0    194377
1.0     35097
Name: count, dtype: int64

In [16]:
#Class distribution (approach adapted from Tutorials)
print(df["Diabetes_binary"].value_counts()/len(df)*100)

Diabetes_binary
0.0    84.705457
1.0    15.294543
Name: count, dtype: float64


Observation: After removing the duplicates, the dataset now contains 229,474 records, mostly the duplicated records belong to class 0 (no diabetes), which could mean that the respondents answered "no" to most health conditions like not having high pressure, no stroke, and that answer is identical, whereas the respondents in class 1 (diabetes)), they often have combinations of conditions.With those results, I can see that the percentage of classes are not well balanced, approximately 84.7% belongs to class 0 and 15.3% to class 1.

In [15]:
df.describe()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
count,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.00000,229474.000000,229474.000000,229474.000000,229474.000000,...,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000,229474.000000
mean,0.152945,0.454343,0.441640,0.959481,28.687507,0.46580,0.044816,0.103336,0.733042,0.612675,...,0.946011,0.092921,2.601820,3.509866,4.681219,0.185751,0.439087,8.085068,4.979741,5.888615
std,0.359936,0.497912,0.496584,0.197173,6.789204,0.49883,0.206899,0.304398,0.442371,0.487140,...,0.225996,0.290323,1.064962,7.717643,9.050877,0.388906,0.496277,3.094451,0.992989,2.092888
min,0.000000,0.000000,0.000000,0.000000,12.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,1.000000,24.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,6.000000,4.000000,4.000000
50%,0.000000,0.000000,0.000000,1.000000,27.000000,0.00000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000,8.000000,5.000000,6.000000
75%,0.000000,1.000000,1.000000,1.000000,32.000000,1.00000,0.000000,0.000000,1.000000,1.000000,...,1.000000,0.000000,3.000000,2.000000,4.000000,0.000000,1.000000,10.000000,6.000000,8.000000
max,1.000000,1.000000,1.000000,1.000000,98.000000,1.00000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,5.000000,30.000000,30.000000,1.000000,1.000000,13.000000,6.000000,8.000000
